In [1]:
import sys 
print(sys.executable)

/Users/stevey/cpsc330arm/bin/python


In [2]:
# python -m pip install -r requirements.txt

import os
import logging
import calendar
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from typing import Iterable, Optional
from pybaseball import statcast


import pyarrow.parquet as pq


import numpy as np


import duckdb

import requests 
import json 

### STEPS
1. get data from pybaseball / MLB API 
2. Parquet on Google Drive 
3. DuckDB reads Parquet 
4. staging / processed / feature tables 

In [3]:
# Google Drive for parquet storage
# read root from .env file
load_dotenv()

logging.basicConfig(
    level=logging.INFO, # shows INFO, WARNING, and ERROR messages
    format="[%(asctime)s] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

YEARS = range(2015, 2026)
MONTHS = [3, 4, 5, 6, 7, 8, 9, 10]
SOURCE_NAME = "statcast"

DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])
RAW_ROOT = DRIVE_ROOT / "data" / "raw" / SOURCE_NAME

DEFAULT_REDO = False
DEFAULT_CONTINUE_ON_ERROR = True


In [14]:
# check file status 
check_df = pd.DataFrame(
    columns=["year", "month", "exists", "nrows", "size_mb"]
)

ls = [] 
for year in YEARS:
    for month in MONTHS:
        filepath = RAW_ROOT / f"year={year}" / f"month={month:02d}" / f"statcast_{year}_{month:02d}.parquet"
        if filepath.exists():
            nrows = pq.ParquetFile(filepath).metadata.num_rows
            size_mb = filepath.stat().st_size / (1024 * 1024)
        else:
            nrows = 0
            size_mb = 0

        ls.append({
            "year": year,
            "month": month,
            "exists": filepath.exists(),
            "nrows": nrows,
            "size_mb": size_mb
        })

check_df = pd.DataFrame(ls).reset_index(drop=True)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(check_df)

,year,month,exists,nrows,size_mb
0,2015,3,False,0,0.000000
1,2015,4,True,94253,10.131628
2,2015,5,True,121910,13.007297
3,2015,6,True,116165,12.430172
4,2015,7,True,108309,11.592992
5,2015,8,True,123528,13.137695
6,2015,9,True,121713,12.983788
7,2015,10,True,16428,1.962010
8,2016,3,False,0,0.000000
9,2016,4,True,104296,11.301394


In [ ]:
# DuckDB for local databasing 
con = duckdb.connect("local_db/bayesball.duckdb")


# TODO: insert path for read_parquet 
con.execute("""
    CREATE OR REPLACE VIEW raw_statcast AS 
    SELECT * 
    FROM read_parquet('.')
""")
